# AWS Cost & Usage Assistant

Uses OpenAI function calling to answer natural-language questions about AWS spend by invoking Cost Explorer and EC2 APIs through boto3.

In [ ]:
import os
import json
import datetime
import boto3
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4o"
openai = OpenAI()

## Verify AWS credentials

Confirm boto3 is picking up credentials and check which IAM identity is active.

In [ ]:
session = boto3.Session()
creds = session.get_credentials()
print("Credential source:", creds.method)

## Tool functions

Each function wraps a boto3 call that the model can invoke via function calling.

In [ ]:
def get_aws_cost(days: int = 7) -> str:
    ce = boto3.client("ce")
    end = datetime.date.today()
    start = end - datetime.timedelta(days=days)
    resp = ce.get_cost_and_usage(
        TimePeriod={"Start": start.isoformat(), "End": end.isoformat()},
        Granularity="DAILY",
        Metrics=["UnblendedCost"],
    )
    return json.dumps(resp["ResultsByTime"])


def get_cost_by_service(days: int = 7) -> str:
    ce = boto3.client("ce")
    end = datetime.date.today()
    start = end - datetime.timedelta(days=days)
    resp = ce.get_cost_and_usage(
        TimePeriod={"Start": start.isoformat(), "End": end.isoformat()},
        Granularity="DAILY",
        Metrics=["UnblendedCost"],
        GroupBy=[{"Type": "DIMENSION", "Key": "SERVICE"}],
    )
    return json.dumps(resp["ResultsByTime"])


def get_s3_storage_cost(days: int = 7) -> str:
    ce = boto3.client("ce")
    end = datetime.date.today()
    start = end - datetime.timedelta(days=days)
    resp = ce.get_cost_and_usage(
        TimePeriod={"Start": start.isoformat(), "End": end.isoformat()},
        Granularity="DAILY",
        Metrics=["UnblendedCost"],
        Filter={"Dimensions": {"Key": "SERVICE", "Values": ["Amazon Simple Storage Service"]}},
        GroupBy=[{"Type": "DIMENSION", "Key": "USAGE_TYPE"}],
    )
    return json.dumps(resp["ResultsByTime"])


def list_ec2_instances() -> str:
    ec2 = boto3.client("ec2")
    resp = ec2.describe_instances()
    instances = [
        {"id": i["InstanceId"], "type": i["InstanceType"], "state": i["State"]["Name"]}
        for r in resp["Reservations"] for i in r["Instances"]
    ]
    return json.dumps(instances)


TOOL_FUNCTIONS = {
    "get_aws_cost": get_aws_cost,
    "get_cost_by_service": get_cost_by_service,
    "get_s3_storage_cost": get_s3_storage_cost,
    "list_ec2_instances": list_ec2_instances,
}

## Tool schema

JSON schema definitions the model uses to decide which function to call and with what arguments.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_aws_cost",
            "description": "Get total AWS unblended cost for the last N days from Cost Explorer",
            "parameters": {
                "type": "object",
                "properties": {"days": {"type": "integer", "description": "days to look back"}},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_cost_by_service",
            "description": "Get AWS unblended cost for the last N days, broken down by service",
            "parameters": {
                "type": "object",
                "properties": {"days": {"type": "integer", "description": "days to look back"}},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_s3_storage_cost",
            "description": "Get S3 storage cost breakdown for the last N days",
            "parameters": {
                "type": "object",
                "properties": {"days": {"type": "integer", "description": "days to look back"}},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_ec2_instances",
            "description": "List all EC2 instances in the account with their type and current state",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
]

## Single tool-call demo

Ask a cost question; the model picks `get_aws_cost` and we execute it once.

In [ ]:
messages = [{"role": "user", "content": "Why did my AWS spend spike this week?"}]

resp = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
msg = resp.choices[0].message

if msg.tool_calls:
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)
        result = TOOL_FUNCTIONS[call.function.name](**args)
        messages.append(msg)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    final = openai.chat.completions.create(model=MODEL, messages=messages)
    print(final.choices[0].message.content)
else:
    print(msg.content)

## Agentic loop

General loop that keeps calling tools until the model responds with plain text.

In [ ]:
messages = [{"role": "user", "content": "Is my AWS spend normal, and what's actually running right now?"}]

while True:
    resp = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    msg = resp.choices[0].message
    messages.append(msg)

    if not msg.tool_calls:
        print(msg.content)
        break

    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)
        result = TOOL_FUNCTIONS[call.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})